# 🧬 Advanced Feature Engineering + Training Pipeline

**Goal**: Improve RMSE from ~26 to lower with advanced features  
**Features**: Velocity, Acceleration, Rolling Stats, Pseudo-IOB  
**Optimization**: Polars Lazy API + XGBoost QuantileDMatrix

In [ ]:
!pip install polars xgboost -U -q

import polars as pl
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
import gc
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error

from google.colab import drive
drive.mount('/content/drive')

print(f"✅ Polars: {pl.__version__}, XGBoost: {xgb.__version__}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"

RAW_TRAIN = f"{BASE_PATH}/raw/metabonet_public_train.parquet"
RAW_TEST = f"{BASE_PATH}/raw/metabonet_public_test.parquet"
PROC_TRAIN = f"{BASE_PATH}/processed/train_ready_for_glucose_prediction.parquet"
PROC_TEST = f"{BASE_PATH}/processed/test_ready_for_glucose_prediction.parquet"
MODEL_OUTPUT = f"{BASE_PATH}/processed/xgboost_glucose_model_v2.json"

os.makedirs(f"{BASE_PATH}/processed", exist_ok=True)

# Keep from raw data
RAW_KEEP = ['id', 'date', 'CGM', 'bolus', 'carbs', 'basal', 'insulin_type_bolus', 'gender', 'age', 'weight']

# Advanced features list (expanded)
FEATURES = [
    # Base
    'CGM', 'bolus', 'carbs', 'basal',
    # Time
    'time_sin', 'time_cos', 'is_weekend',
    # Trend (velocity/acceleration)
    'velocity', 'acceleration',
    'cgm_delta_5min', 'cgm_delta_15min',
    # Rolling statistics
    'rolling_mean_30m', 'rolling_std_1h',
    # Pseudo-IOB (active insulin)
    'rolling_insulin_2h',
    # Demographics
    'age', 'weight',
    'gender_encoded', 'insulin_type_encoded'
]

TARGET = 'CGM_future'
FORECAST_STEPS = 6

print(f"📊 Features ({len(FEATURES)}): {FEATURES}")

## Step 1: Advanced Feature Engineering (Streaming)

In [ ]:
def create_advanced_features(raw_path: str, output_path: str):
    """
    Advanced feature engineering with Polars Lazy API.
    Features: Velocity, Acceleration, Rolling Stats, Pseudo-IOB
    """
    print("="*70)
    print(f"🔧 ADVANCED FEATURE ENGINEERING: {raw_path}")
    print("="*70)
    
    # 1. Scan (lazy)
    print("\n📂 Scanning parquet...")
    q = pl.scan_parquet(raw_path)
    available = q.collect_schema().names()
    keep = [c for c in RAW_KEEP if c in available]
    q = q.select(keep)
    
    # 2. Sort by patient and time
    print("📌 Sorting by [id, date]...")
    q = q.sort(['id', 'date'])
    
    # 3. Filter null CGM & impossible values
    print("🧹 Filtering invalid CGM (null, <30, >500)...")
    q = q.filter(
        pl.col('CGM').is_not_null() &
        (pl.col('CGM') >= 30) &
        (pl.col('CGM') <= 500)
    )
    
    # 4. Fill nulls & cast
    print("🔧 Filling nulls...")
    q = q.with_columns([
        pl.col('bolus').fill_null(0.0).cast(pl.Float32),
        pl.col('carbs').fill_null(0.0).cast(pl.Float32),
        pl.col('basal').fill_null(0.0).cast(pl.Float32),
        pl.col('age').fill_null(-1.0).cast(pl.Float32),
        pl.col('weight').fill_null(-1.0).cast(pl.Float32),
        pl.col('CGM').cast(pl.Float32),
    ])
    
    # 5. Time features
    print("⏰ Creating time features...")
    q = q.with_columns([
        pl.col('date').dt.hour().alias('hour'),
        pl.col('date').dt.weekday().alias('weekday'),
    ])
    q = q.with_columns([
        (2 * np.pi * pl.col('hour') / 24).sin().cast(pl.Float32).alias('time_sin'),
        (2 * np.pi * pl.col('hour') / 24).cos().cast(pl.Float32).alias('time_cos'),
        pl.when(pl.col('weekday') >= 5).then(1.0).otherwise(0.0).cast(pl.Float32).alias('is_weekend'),
    ])
    
    # ============================================================
    # 6. ADVANCED FEATURES - THE SECRET SAUCE
    # ============================================================
    print("\n🧬 Creating ADVANCED features...")
    
    # --- Velocity: CGM - CGM_lag_1 (rate of change) ---
    print("   📈 Velocity (CGM change/5min)...")
    q = q.with_columns(
        (pl.col('CGM') - pl.col('CGM').shift(1).over('id')).cast(pl.Float32).alias('velocity')
    )
    
    # --- Acceleration: Velocity - Velocity_lag_1 (rate of rate) ---
    print("   📈 Acceleration (velocity change)...")
    q = q.with_columns(
        (pl.col('velocity') - pl.col('velocity').shift(1).over('id')).cast(pl.Float32).alias('acceleration')
    )
    
    # --- CGM Deltas ---
    print("   📈 CGM deltas (5min, 15min)...")
    q = q.with_columns([
        (pl.col('CGM') - pl.col('CGM').shift(1).over('id')).cast(pl.Float32).alias('cgm_delta_5min'),
        (pl.col('CGM') - pl.col('CGM').shift(3).over('id')).cast(pl.Float32).alias('cgm_delta_15min'),
    ])
    
    # --- Rolling Mean 30min (6 periods): Trend smoothing ---
    print("   📊 Rolling Mean 30min (noise reduction)...")
    q = q.with_columns(
        pl.col('CGM').rolling_mean(window_size=6).over('id').cast(pl.Float32).alias('rolling_mean_30m')
    )
    
    # --- Rolling Std 1hr (12 periods): Volatility ---
    print("   📊 Rolling Std 1hr (volatility)...")
    q = q.with_columns(
        pl.col('CGM').rolling_std(window_size=12).over('id').cast(pl.Float32).alias('rolling_std_1h')
    )
    
    # --- Pseudo-IOB: Rolling SUM of bolus over 2hrs (24 periods) ---
    print("   💉 Pseudo-IOB (rolling insulin 2hr)...")
    q = q.with_columns(
        pl.col('bolus').rolling_sum(window_size=24).over('id').cast(pl.Float32).alias('rolling_insulin_2h')
    )
    
    # ============================================================
    # 7. CATEGORICAL ENCODING
    # ============================================================
    print("\n🏷️ Encoding categoricals...")
    if 'gender' in keep:
        q = q.with_columns(
            pl.when(pl.col('gender').str.to_lowercase() == 'male').then(1.0)
            .when(pl.col('gender').str.to_lowercase() == 'female').then(0.0)
            .otherwise(-1.0).cast(pl.Float32).alias('gender_encoded')
        )
    else:
        q = q.with_columns(pl.lit(-1.0).cast(pl.Float32).alias('gender_encoded'))
    
    if 'insulin_type_bolus' in keep:
        q = q.with_columns(
            pl.when(pl.col('insulin_type_bolus').str.to_lowercase().str.contains('rapid')).then(1.0)
            .when(pl.col('insulin_type_bolus').str.to_lowercase().str.contains('long')).then(2.0)
            .when(pl.col('insulin_type_bolus').str.to_lowercase().str.contains('mix')).then(3.0)
            .otherwise(0.0).cast(pl.Float32).alias('insulin_type_encoded')
        )
    else:
        q = q.with_columns(pl.lit(0.0).cast(pl.Float32).alias('insulin_type_encoded'))
    
    # ============================================================
    # 8. TARGET CREATION
    # ============================================================
    print("\n🎯 Creating target (CGM_future = shift -6)...")
    q = q.with_columns(
        pl.col('CGM').shift(-FORECAST_STEPS).over('id').cast(pl.Float32).alias('CGM_future')
    )
    q = q.filter(pl.col('CGM_future').is_not_null())
    
    # ============================================================
    # 9. FILTER STATIC CGM (Sensor Error Detection)
    # ============================================================
    print("\n🔍 Filtering static CGM readings (sensor errors)...")
    # If velocity is 0 for too long, it's likely a sensor error
    # We remove rows where velocity == 0 AND acceleration == 0
    q = q.filter(
        ~((pl.col('velocity') == 0) & (pl.col('acceleration') == 0))
    )
    
    # ============================================================
    # 10. FINAL SELECTION & SAVE
    # ============================================================
    print("\n📋 Final column selection...")
    final_cols = FEATURES + [TARGET]
    q = q.select(final_cols)
    q = q.drop_nulls()
    
    print(f"\n💾 Streaming to: {output_path}")
    q.sink_parquet(output_path)
    
    # Verify
    if os.path.exists(output_path):
        size = os.path.getsize(output_path) / (1024**3)
        count = pl.scan_parquet(output_path).select(pl.len()).collect().item()
        print(f"✅ Saved: {size:.2f} GB, {count:,} rows")
    
    return output_path

In [ ]:
# Process TRAIN data
create_advanced_features(RAW_TRAIN, PROC_TRAIN)

In [ ]:
# Process TEST data
create_advanced_features(RAW_TEST, PROC_TEST)

## Step 2: Training with Early Stopping

In [ ]:
print("="*70)
print("🚀 TRAINING WITH ADVANCED FEATURES")
print("="*70)

# Load train
print("\n📂 Loading train data...")
df_train = pl.read_parquet(PROC_TRAIN)
print(f"✅ Shape: {df_train.shape}")

X_train = df_train.select(FEATURES).to_numpy().astype(np.float32)
y_train = df_train.select(TARGET).to_numpy().ravel().astype(np.float32)
del df_train; gc.collect()

# Load test for early stopping
print("\n📂 Loading test data (for early stopping)...")
df_test = pl.read_parquet(PROC_TEST)
X_test = df_test.select(FEATURES).to_numpy().astype(np.float32)
y_test = df_test.select(TARGET).to_numpy().ravel().astype(np.float32)
del df_test; gc.collect()

print(f"   Train: {X_train.shape}, Test: {X_test.shape}")

# Create QuantileDMatrix
print("\n⚡ Creating QuantileDMatrix...")
dtrain = xgb.QuantileDMatrix(X_train, label=y_train, feature_names=FEATURES, max_bin=256)
dtest = xgb.QuantileDMatrix(X_test, label=y_test, feature_names=FEATURES, max_bin=256)
del X_train, y_train, X_test; gc.collect()

# Parameters
params = {
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': 'cuda',
    'max_depth': 8,  # Increased for complex features
    'learning_rate': 0.03,  # Lower for better convergence
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'reg_alpha': 0.1,  # L1 regularization
    'reg_lambda': 1.0,  # L2 regularization
    'seed': 42
}

print("\n📋 Parameters:")
for k, v in params.items():
    print(f"   {k}: {v}")

# Train with early stopping
print("\n⏳ Training (2000 rounds, early stopping=50)...")
model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=2000,
    evals=[(dtrain, 'train'), (dtest, 'test')],
    early_stopping_rounds=50,
    verbose_eval=100
)

print(f"\n✅ Best iteration: {model.best_iteration}")
print(f"✅ Best score: {model.best_score:.4f}")

# Save
model.save_model(MODEL_OUTPUT)
print(f"\n💾 Model saved: {MODEL_OUTPUT}")

del dtrain; gc.collect()

## Step 3: Evaluation

In [ ]:
print("="*70)
print("📊 EVALUATION")
print("="*70)

# Predict on test
y_pred = model.predict(dtest)
del dtest; gc.collect()

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
abs_errors = np.abs(y_test - y_pred)
within_15 = np.mean(abs_errors <= 15) * 100
within_20 = np.mean(abs_errors <= 20) * 100

print("\n" + "="*50)
print("🎯 RESULTS (Advanced Features)")
print("="*50)
print(f"  RMSE:           {rmse:.4f} mg/dL")
print(f"  MAE:            {mae:.4f} mg/dL")
print(f"  Clinical ±15:   {within_15:.2f}%")
print(f"  Clinical ±20:   {within_20:.2f}%")
print("="*50)

# Improvement check
old_rmse = 26.0
improvement = ((old_rmse - rmse) / old_rmse) * 100
print(f"\n📈 Improvement: {improvement:.2f}% (from {old_rmse} to {rmse:.2f})")

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Time series
n = 200
axes[0,0].plot(range(n), y_test[:n], 'g-', lw=2, label='Actual')
axes[0,0].plot(range(n), y_pred[:n], 'r--', lw=2, label='Predicted')
axes[0,0].set_title(f'Actual vs Predicted (RMSE: {rmse:.2f})')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# 2. Scatter
sample = min(50000, len(y_test))
axes[0,1].scatter(y_test[:sample], y_pred[:sample], alpha=0.2, s=2)
axes[0,1].plot([0,400], [0,400], 'r--', lw=2)
axes[0,1].set_title('Scatter'); axes[0,1].grid(alpha=0.3)

# 3. Error histogram
errors = y_test[:sample] - y_pred[:sample]
axes[1,0].hist(errors, bins=100, color='purple', alpha=0.7)
axes[1,0].axvline(0, color='red', ls='--')
axes[1,0].axvline(15, color='green', ls=':', label='±15')
axes[1,0].axvline(-15, color='green', ls=':')
axes[1,0].set_title('Error Distribution'); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

# 4. Feature importance (top 15)
imp = model.get_score(importance_type='gain')
sorted_imp = sorted(imp.items(), key=lambda x: x[1], reverse=True)[:15]
axes[1,1].barh([x[0] for x in sorted_imp[::-1]], [x[1] for x in sorted_imp[::-1]], color='teal')
axes[1,1].set_title('Top 15 Feature Importance')

plt.tight_layout()
plt.show()

print("\n🎉 COMPLETE!")